# Milestone 4 Lab, notebook 5 of 7: A trace of the analyst

## Where we are

| Notebook | What you do |
|---|---|
| 1. A figure, traced by hand | Find every record behind one figure yourself, and add them up |
| 2. Run records | Give every pipeline run an id and a record, and keep every run |
| 3. Lineage | Save which records made each figure, and prove it adds up |
| 4. Two runs compared | Find what changed between two runs: settings, inputs, figures, records |
| **5. A trace of the analyst** (you are here) | Give the analyst the new tools, and record every step it takes |
| 6. Scoring an agent | Measure an agent with a confusion matrix, kappa, and the cost of mistakes |
| 7. Two experiments | Run two versions of the analyst on the same questions, and compare the scores |

Model requests in this notebook: **about 15**. A full run of this lab costs about 5 US cents.

## What this notebook is for

Notebooks 1 to 4 made the pipeline remember what it did. No AI was involved.

Now the AI comes in, in two ways:

1. **The analyst uses the new records.** It gets a tool that compares two runs, and its evidence tool now reads lineage. So it can answer "where did this number come from?" and "why did it change?".
2. **We start remembering what the analyst does.** An agent takes several steps to reach an answer. A record of those steps is called a **trace**. You build one by hand, then let Langfuse keep them.

Lineage and a trace are the same idea. Lineage records how a **figure** was made. A trace records how an **answer** was made.

> This notebook needs `OPENROUTER_API_KEY` and the three Langfuse settings in your `.env` file. The milestone's README says how to get the Langfuse ones.

In [ ]:
import pandas as pd
from lab_helpers import connect, connect_langfuse, load_result, show_answer, show_doc

## Step 1: Start from the real history

Notebooks 2 to 4 worked in a lab folder. The analyst reads the real warehouse folder. `make_history` clears it and makes the two runs the milestone starts with: `RUN-0001` (duplicates counted) and `RUN-0002` (duplicates set aside).

In [ ]:
from retail_profit_stack.make_history import make_history

for run in make_history():
    print(run.run_id, "-", run.record["note"])

## Step 2: The simple approach first

Before giving the analyst any new tool, try the simplest thing. Give a model the two figures and ask it why they differ. It has nothing else to go on.

In [ ]:
client, MODEL = connect()

In [ ]:
prompt = """The trail water bottle's contribution profit for October 2026 was $5,188.20 in our
previous calculation. In today's calculation it is $5,156.94.

Why did it change? Answer in three sentences."""

reply = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=1500,
)

print(reply.choices[0].message.content)

Read the reply. The model can subtract the two figures. It cannot know the reason, because the reason is not in the prompt. A careful model says so. A less careful one offers likely causes, such as returns or cost changes. Those are guesses, and a leader may take a guess for a fact.

This is the same position you were in at the end of notebook 1: a guess, where a record is needed. If the cell failed with a network error, run it again.

## Step 3: The two tools that read the records

Here is the design.

In [ ]:
show_doc("LLD.md", "6. The tools")

A tool is a plain Python function, so we can call it ourselves. First `compare_runs`, for the figure we care about.

In [ ]:
from retail_profit_stack import tools

comparison = tools.compare_runs("2026-10", "trail water bottle")

print("Compared:", comparison.earlier_run, "with", comparison.later_run)
print("Settings changed:", comparison.settings_changed)
print("Inputs changed:  ", comparison.inputs_changed)
print("Code changed:    ", comparison.code_changed)

In [ ]:
comparison.figures_changed

In [ ]:
pd.DataFrame(comparison.records_changed)

That is notebook 4, returned by one function. The before, the after, and the change are all worked out by code. The model will only have to repeat them.

Now the evidence tool. In milestone 3 it put together a few sample records each time it was asked. It now reads the lineage that the run saved.

In [ ]:
evidence = tools.get_evidence("2026-10", "trail water bottle")

print("Run:              ", evidence.run_id)
print("Records by table: ", evidence.record_counts)
print("Total records:    ", evidence.total_records)
print("Records set aside:", evidence.set_aside_count)

In [ ]:
evidence.steps

754 records, the number you found by hand in notebook 1. The tool still returns only five of the records themselves, to keep its reply short. The counts cover all of them.

## Step 4: Ask the analyst

`run_analyst` is the same function as in milestone 3. Its instructions have two new rules, and it has seven tools where it had six.

In [ ]:
from retail_profit_stack.agent import RUN_TOOL_RULE

print(RUN_TOOL_RULE)

In [ ]:
from retail_profit_stack.agent import run_analyst

QUESTION = "Why did the trail water bottle's October 2026 contribution profit change since the last run?"

run = run_analyst(QUESTION)

show_answer(run.answer)

Compare it with the answer key you made by hand in notebook 1.

In [ ]:
lineage_key = load_result("answer_key_lineage", required_from="notebook 1")

by_hand = {
    "earlier_run_value": lineage_key["earlier_contribution_profit"],
    "value": lineage_key["contribution_profit"],
    "run_change": lineage_key["change"],
}

rows = []
for kind, correct in by_hand.items():
    said = [figure.value for figure in run.answer.figures if figure.kind == kind]
    rows.append({
        "figure": kind,
        "by hand (notebook 1)": correct,
        "the analyst said": said[0] if said else "not given",
        "right?": "yes" if said and abs(said[0] - correct) < 0.005 else "NO",
    })

pd.DataFrame(rows)

The verifier checked the answer too. It now has nine checks. The last one, `run_cited`, is new: an answer must name the run its figures came from.

In [ ]:
print("The answer names run:", run.answer.run_id)

pd.DataFrame([check.model_dump() for check in run.verification.checks])

If a check failed, run step 4 again. The analyst is good, not perfect, and that is what the next two notebooks measure.

## Step 5: What did the analyst actually do?

You saw the answer. You did not see how the analyst got there: which tools it called, in what order, and how long each step took. When an answer is wrong, the cause is in one of those steps.

LangChain can tell us. It accepts a **callback handler**: an object with methods that LangChain calls by itself when something happens. `on_tool_start` is called when a tool call starts, `on_llm_end` when a model call ends, and so on.

Here is a small one. Each method adds one line to a list.

In [ ]:
import time

from langchain_core.callbacks import BaseCallbackHandler


class StepList(BaseCallbackHandler):
    """Writes down each step of an agent run."""

    def __init__(self):
        self.steps = []
        self.clock = time.time()

    def add(self, what, detail):
        seconds = round(time.time() - self.clock, 2)
        self.steps.append({"at (seconds)": seconds, "what": what, "detail": detail[:150]})

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.add("model call starts", f"{len(messages[0])} messages sent")

    def on_llm_end(self, response, **kwargs):
        message = response.generations[0][0].message
        asked_for = [call["name"] for call in message.tool_calls]
        self.add("model call ends", f"the model asks for: {asked_for}")

    def on_tool_start(self, serialized, input_str, **kwargs):
        self.add("tool call starts", f"{serialized['name']} {input_str}")

    def on_tool_end(self, output, **kwargs):
        self.add("tool call ends", str(getattr(output, "content", output)))

To use it we build the analyst agent ourselves, the way `run_analyst` does inside, and pass the handler in `callbacks`.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

from retail_profit_stack.agent import SYSTEM_PROMPT, make_model, make_tools
from retail_profit_stack.schemas import AnalystAnswer

tool_log = []
agent = create_agent(
    model=make_model(),
    tools=make_tools(tool_log),
    system_prompt=SYSTEM_PROMPT + RUN_TOOL_RULE,
    response_format=ToolStrategy(AnalystAnswer),
)

In [ ]:
step_list = StepList()

messages = [{"role": "user", "content": QUESTION}]
result = agent.invoke({"messages": messages}, {"callbacks": [step_list]})

pd.set_option("display.max_colwidth", 150)
pd.DataFrame(step_list.steps)

**That table is a trace.** Read it from the top:

- A model call starts and ends. The model asks for one or two tools.
- Each tool call starts and ends. You can see its arguments and the start of what it returned.
- Another model call. This time the model asks for `AnalystAnswer`: that is how it hands in its structured answer.

A trace is nothing more than this: a list of steps, each with a start, an end, an input, and an output.

Production keeps the same list for every run, with the tokens and cost of each model call. It is in `run.steps`.

In [ ]:
pd.DataFrame(run.steps)

In [ ]:
print("This answer cost:", run.cost, "US dollars, and took", run.seconds, "seconds")

## Step 6: Langfuse

Our list has a weakness: it lives in this notebook. Close the notebook and it is gone. Nobody else can see it, and you cannot search last week's answers.

**Langfuse** solves that. Here it is in plain words, from the HLD.

In [ ]:
show_doc("HLD.md", "8. Langfuse, in plain words")

In [ ]:
langfuse = connect_langfuse()

Langfuse ships its own callback handler. It has the same methods as your `StepList`. Where yours adds a line to a list, Langfuse's sends the step to the Langfuse website.

`run_analyst(..., trace=True)` adds that handler. Nothing else changes.

In [ ]:
traced = run_analyst(QUESTION, trace=True)
langfuse.flush()

show_answer(traced.answer)

`flush` tells Langfuse to send anything it is still holding. The run now carries the id of its trace.

In [ ]:
print("Trace id:", traced.trace_id)
print("Open it: ", langfuse.get_trace_url(trace_id=traced.trace_id))

**Open that link.** It can take a few seconds to appear. Things to look at:

- **The tree on the left.** One row per step, indented to show what ran inside what. It is your `StepList` table drawn as a tree.
- **Click a model call** (Langfuse calls it a *generation*). You see the full prompt, the reply, the tokens, the cost, and the time.
- **Click a tool call.** You see its arguments and its full result.
- **The tags at the top.** One of them is `RUN-0002`. We label each trace with the pipeline run its tools read.

You can also open **Tracing** in the Langfuse menu to see every trace, newest first.

## Step 7: From an answer back to the records

Put the pieces of this milestone in one line:

> **answer → trace → pipeline run → lineage → source records**

In [ ]:
print("The answer:      ", traced.answer.headline)
print("Its trace:       ", traced.trace_id)
print("The pipeline run:", traced.run_id)
print("The lineage:     ", evidence.total_records, "records in", list(evidence.record_counts))
print("One record:      ", evidence.records[0].description)

Every link in that chain is saved. A person can start from any answer the analyst ever gave and walk back to the rows in the source tables.

## Step 8: The other new question

Ask the analyst where a figure came from, with tracing on.

In [ ]:
WHERE_QUESTION = "Where did the trail water bottle's October 2026 contribution profit come from?"

where = run_analyst(WHERE_QUESTION, trace=True)
langfuse.flush()

show_answer(where.answer)

In [ ]:
print("Tools called:", [call.name for call in where.tool_log])
print("Verified:    ", where.verification.passed)
print("Open it:     ", langfuse.get_trace_url(trace_id=where.trace_id))

## What we now know

- With only two figures to go on, a model can subtract and can only **guess** the reason.
- `compare_runs` gives the analyst the reason from the records. Code works out every amount; the analyst repeats and explains.
- The evidence tool now reads **lineage**: it reports all 754 records behind the figure, and the run they belong to.
- A **trace** is a list of steps, each with a start, an end, an input, and an output. We built one in about twenty lines.
- **Langfuse** keeps traces on a website, where they outlive the notebook and can be opened by anyone on the team.
- The chain is complete: answer → trace → pipeline run → lineage → source records.

**Next:** notebook 6. One good answer proves little. Before running the analyst on every question, we need a better way to score it than right or wrong.